# 03 A*：启发搜索（可采纳、一致、与 Dijkstra 的一行之差）


A*（Hart, Nilsson, Raphael, 1968）是带启发的 Dijkstra。百科式写法如下。

---

## 1. 定义

对每个顶点维护

\[
f(n)=g(n)+h(n)
\]

- \(g(n)\)：从起点到 n 的已知代价（与 Dijkstra 相同）
- \(h(n)\)：从 n 到终点的 **估计**（启发）
- 优先队列按 **f** 取最小，而不是按 g

直观：g 管「已经花了多少」（避免贪地乱走），h 管「还剩多少」（避免往远离终点的方向扩）。

特例：

| h | 算法 |
|---|------|
| \(h=0\) | Dijkstra / 一致代价搜索 UCS |
| 只用 h、忽略 g | 贪心最佳优先，**不必最优** |
| 等权 + 队列不用 f | BFS |

---

## 2. 可采纳（admissible）

\(h\) **可采纳**：对任意 n，\(h(n)\le h^*(n)\)，其中 \(h^*\) 是 n 到目标的真实最短剩余。即 **从不高估**。另有 \(h(\text{goal})=0\)。

**定理：** h 可采纳 ⇒ A* 返回的路径最优（图有限、边权非负；见下节「关闭」细节）。

证明梗概：设 A* 返回路径 P 代价 \(w(P)\)，最优为 \(w^*\)。若 \(w(P)>w^*\)，看最优路上第一个还在 open 里的点 n。因 h 可采纳，\(f(n)=g^*(n)+h(n)\le w^* < w(P)\)。A* 应先扩展 n 而不是把 P 的终点弹出。矛盾。

构造可采纳 h 的标准办法：解一个 **放宽问题** 的最优剩余。例如网格去掉墙，剩余就是曼哈顿距离。

---

## 3. 一致 / 单调（consistent / monotonic）

\(h\) **一致**：对每条边 \(u\to v\)，

\[
h(u)\le w(u,v)+h(v)
\]

即三角不等式。一致 ⇒ 可采纳。

一致时：一个点 **弹出 open 之后** g 不会再变小，可以放进 closed **永不重开**。  
只可采纳不一致时：可能必须把已关闭点重新放回 open（Wikipedia 伪代码备注）。惰性堆 + `if current in done: continue` 在不一致时 **可能不最优**。本课网格曼哈顿是一致的，关闭即可。

---

## 4. 网格上的 h

四连通、每步代价 1：

\[
h_{\mathrm{man}}((x,y),(x',y'))=|x-x'|+|y-y'|
\]

一步最多让曼哈顿减 1，故 \(h\le h^*\)，且对边权 1 一致。

若边权是欧氏长度、又可任意方向走，欧氏距离 \(\sqrt{dx^2+dy^2}\) 可采纳。四连通却用欧氏：仍可采纳（欧氏 ≤ 曼哈顿 ≤ 真实四连通剩余），但偏乐观，扩展会多一点。

**不可采纳例子：** \(h=50\) 常数，或 \(h=10\times\) 曼哈顿。A* 会过分往终点冲，可能错过真最短。

---

## 5. 伪代码

与 Dijkstra 只有 **堆键** 和 **松弛后的 f** 不同：

```
f[s] ← h(s)
push (f[s], s)
while heap:
    u ← pop 最小 f
    if u 已关闭: continue
    关闭 u
    if u = t: return 路径
    for v in 邻居(u):
        alt ← g[u] + w(u,v)
        if alt < g[v]:
            g[v] ← alt
            came_from[v] ← u
            push (alt + h(v), v)     # 唯一多出来的 +h(v)
```

时间仍取决于堆，最坏与 Dijkstra 同阶；h 越好，实际扩展越少。最坏启发 h=0 时退回 Dijkstra 的扩展集。

---

## 6. 手算要点

墙在 x=3、y=0..6，缺口 (3,7)。S=(0,0)，G=(7,7)。\(h(S)=14\)。

A* 会优先扩展 f 小的：同样 g 时更靠近 G（h 小）的先出。墙挡住直线，仍必须先绕到 y=7，但墙左侧向下的点 f 大，会少扩展。下一格代码会打印每个弹出点的 g/h/f。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


print("h(S,G) =", manhattan((0, 0), (7, 7)))
print("缺口 (3,7) 的 h =", manhattan((3, 7), (7, 7)))
print("墙左下 (0,7) 的 h =", manhattan((0, 7), (7, 7)), "（离 G 仍远，f 会偏大）")


## 7. 从 Dijkstra 改 A*：只改出队键

把 `heappush(heap, (tentative, ...))` 改成 `(tentative + h(nxt, goal), ...)`。第一次 push 起点用 `h(start,goal)`。其余松弛、came_from、惰性跳过，一字不改。


In [ ]:
import heapq
from lib.maps import grid_wall


def reconstruct(came_from, goal):
    node = goal
    path = [node]
    while came_from[node] is not None:
        node = came_from[node]
        path.append(node)
    path.reverse()
    return path


def astar(graph, start, goal, h=manhattan, verbose=False):
    heap = []
    counter = 0
    g_score = {start: 0.0}
    came_from = {start: None}
    heapq.heappush(heap, (h(start, goal), counter, start))
    closed = set()
    expanded = 0
    while heap:
        _f, _, current = heapq.heappop(heap)
        if current in closed:
            continue
        closed.add(current)
        expanded += 1
        g_cur = g_score[current]
        if verbose:
            hn = h(current, goal)
            print(f"弹出 {current}  g={g_cur:.0f}  h={hn:.0f}  f={g_cur+hn:.0f}")
        if current == goal:
            return reconstruct(came_from, goal), g_cur, expanded
        for nxt, cost in graph.neighbors(current):
            tentative = g_cur + cost
            if tentative < g_score.get(nxt, float("inf")):
                g_score[nxt] = tentative
                came_from[nxt] = current
                counter += 1
                # 与 Dijkstra 的唯一差别：键是 g+h 不是 g
                heapq.heappush(heap, (tentative + h(nxt, goal), counter, nxt))
    return None, float("inf"), expanded


graph = grid_wall()
path, cost, n_exp = astar(graph, (0, 0), (7, 7), verbose=True)
print("代价", cost, "扩展", n_exp)


看打印：f 小的点先弹出。墙右边的点要等绕过缺口之后 g 才有限。

---

## 8. 实验：h=0、曼哈顿、胡乱 h


In [ ]:
from lib.search import astar as astar_lib, dijkstra

d = dijkstra(graph, (0, 0), (7, 7), record=False)
a = astar_lib(graph, (0, 0), (7, 7), record=False)
a0 = astar(graph, (0, 0), (7, 7), h=lambda p, q: 0)
bad = astar(graph, (0, 0), (7, 7), h=lambda p, q: 50)

print("Dijkstra     代价", d.cost, "扩展", d.expanded)
print("A* 曼哈顿    代价", a.cost, "扩展", a.expanded, "应==Dijkstra代价")
print("A* h=0       代价", a0[1], "扩展", a0[2], "应接近 Dijkstra 扩展")
print("A* h=50 乱估 代价", bad[1], "扩展", bad[2], "可能 > 最优")


库 `lib.search.best_first` 用参数 `weight`：键是 `g + weight*h`。`weight=1` 即本课 A*。源码里搜 `weight * heuristic` 那一行。


In [ ]:
from lib.studio.widget import PlannerStudio
from lib.search import astar as astar_lib

r = astar_lib(graph, (0, 0), (7, 7))
studio = PlannerStudio()
studio.show_search({**graph.to_studio(), "start": [0, 0], "goal": [7, 7]}, r, "A*")
studio


单步：波前不是 Dijkstra 的菱形匀速扩散，而是被 h 拉成朝向 G 的一瓣（有墙时会变形）。

## 条目小结

| 项目 | 内容 |
|------|------|
| 公式 | \(f=g+h\) |
| 最优 | h 可采纳（关闭策略在一致时最简单） |
| 网格 h | 四连通 → 曼哈顿 |
| 实现差 | 相对 Dijkstra 只改堆键 |
| 退化 | h=0 → Dijkstra；等权 FIFO → BFS |

## 练习

1. 证明四连通、步长 1 时曼哈顿可采纳。
2. 一致条件 \(h(u)\le 1+h(v)\) 对相邻格是否成立？曼哈顿差最多 1。
3. 用 `h=0` 跑墙图，扩展次数应与 Dijkstra 同阶。
